<a href="https://colab.research.google.com/github/Sebastianelli-Nicola/Traffic-Driven-EV-Queuing-Predictor/blob/main/experimental_protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecasting del Traffico e Occupazione Stazioni di Ricarica EV
**Pipeline Sperimentale e Valutazione dei Modelli Predittivi**

Questo notebook implementa l'architettura di Machine Learning progettata per la previsione a breve termine del traffico veicolare. L'infrastruttura di codice è strutturata per rispondere alle specifiche delineate nel piano di progetto, suddividendo il lavoro in due fasi operative principali.

---

## Fase 2: Definizione del Protocollo Sperimentale

In questa prima fase, l'approccio metodologico ai dati è stato completamente ridefinito per simulare in modo accurato un ambiente di produzione reale, superando i limiti dei test statici.

*   **Implementazione della Sliding Window:** Il tradizionale partizionamento statico dei dati (80% per il training, 20% per il test) è stato abbandonato. Il sistema adotta ora un approccio dinamico basato sulla tecnica della *finestra scorrevole* (Sliding Window).
*   **Impostazione dei Test Indipendenti:** L'infrastruttura isola un periodo di prova finale (es. le ultime due settimane disponibili). All'interno di tale periodo, ogni singola giornata costituisce un test indipendente. Per addestrare il modello in vista di ogni test, il sistema sfrutta quasi tutti i dati storici a disposizione.
*   **Aggregazione Temporale:** L'obiettivo non è una previsione generica, ma puntuale. Per questo motivo, l'ingestione dei dati, l'addestramento e i risultati dei test sono strutturati e aggregati in **slot temporali esatti della durata di 30 minuti**.

---

## Fase 3: Esecuzione Computazionale e Calcolo Metriche

Questa sezione rappresenta il cuore operativo. È progettata per eseguire computazioni intensive in background (elaborazione offline) ed estrarre metriche per un futuro deployment del sistema.

*   **Selezione dei Modelli Predittivi:** Per ora sono stati inseriti solo alcuni modelli (Random Forest, SARIMA, LSTM). A queste è stata integrata la sperimentazione di **Neural Prophet**, un'architettura moderna basata su PyTorch.
*   **Elaborazione Offline Automatizzata:** Le routine di addestramento e validazione (Evaluation Loop) sono incapsulate in script automatizzati. Questo permette l'esecuzione autonoma e non supervisionata di interi batch di calcolo.
*   **Tracciamento delle Tempistiche:** Il sistema registra puntualmente le performance hardware e software. Viene misurato il tempo di *Training* (calcolo pesante eseguito offline) e la latenza di *Predizione*. Quest'ultima metrica è critica: nel sistema a regime, il calcolo predittivo dovrà risultare estremamente veloce per poter essere ricalcolato continuamente in modalità online.
*   **Analisi del Decadimento dell'Errore nel Tempo:** Per determinare il ciclo di vita utile del modello, l'orizzonte di previsione massimo viene spinto fino a **2 o 3 giorni**. Il codice calcola e registra l'errore commesso per *ogni singolo slot di 30 minuti*. Mappando come si modifica tale errore all'allontanarsi dal momento della previsione, è possibile stabilire oggettivamente l'affidabilità temporale del sistema e determinare la frequenza ideale per il riaddestramento.

## 1: Ingestion & Pre-Processing
Questa sezione si occupa di importare i dati grezzi da Google Drive, isolare le misurazioni della singola stazione e aggregare i dati in slot temporali esatti da **30 minuti**. Vengono inoltre estratte le feature temporali di base (ora, giorno, weekend) che i modelli utilizzeranno per apprendere i pattern comportamentali.

In [ ]:
# Installa le librerie che non sono native in Google Colab
!pip install neuralprophet

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from google.colab import drive
import warnings

# Ignoriamo i warning non critici per mantenere l'output pulito durante le stampe
warnings.filterwarnings('ignore')

# =========================================================================
# 1. SETUP DELL'AMBIENTE E CONNESSIONE DATI
# =========================================================================
# Montiamo Google Drive per poter leggere e scrivere i file
drive.mount('/content/drive')

# Definiamo i percorsi base (PATH) dove si trovano i file
PATH = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/'
PATH_RISULTATI = '/content/drive/MyDrive/Colab Notebooks/TIROCINIO/'
FILE_NAME = 'traffico_r.csv'

# Creiamo una cartella dedicata SOLO ai risultati
RESULTS_DIR = os.path.join(PATH_RISULTATI, 'Risultati_Esperimenti/')

# Verifica se la cartella esiste già, altrimenti la crea
if not os.path.exists(RESULTS_DIR):
    os.makedirs(RESULTS_DIR)
    print(f" Cartella risultati creata: {RESULTS_DIR}")
else:
    print(f" Cartella risultati trovata: {RESULTS_DIR}")


# =========================================================================
# 2. FUNZIONI DI CARICAMENTO E PRE-PROCESSING
# =========================================================================
def carica_dataset_completo(nome_file):
    """
    Legge il file CSV una sola volta in memoria centrale.
    Questo evita di far crashare la RAM ricaricandolo per ogni stazione.
    """
    print("Lettura del dataset completo in corso...")
    percorso_completo = os.path.join(PATH, nome_file)

    # Controllo di sicurezza: verifichiamo che il file esista prima di leggerlo
    if not os.path.exists(percorso_completo):
        raise FileNotFoundError(f"File {nome_file} non trovato nel percorso {PATH}")

    # Caricamento tramite pandas e conversione immediata della colonna 'timestamp' in formato Datetime
    df = pd.read_csv(percorso_completo)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df

def pre_processing_stazione(df_completo, id_stazione):
    """
    Estrae i dati di una specifica stazione dal dataset globale,
    li regolarizza a slot di 30 minuti e genera le feature temporali.
    """
    # 1. Filtraggio: teniamo solo le righe dove id_stazione corrisponde a quello richiesto
    df_stazione = df_completo[df_completo['id_stazione'] == id_stazione].copy()

    # 2. Impostiamo il timestamp come Indice del dataframe e lo ordiniamo cronologicamente
    df_stazione.set_index('timestamp', inplace=True)
    df_stazione.sort_index(inplace=True)

    # 3. Aggregazione temporale (Resampling)
    # Raggruppa i dati in blocchi esatti di 30 minuti calcolando la media del traffico ('differenza')
    df_resampled = df_stazione[['differenza']].resample('30min').mean()

    # Se il resampling crea dei 'buchi' (NaN) perché in quei 30 min non c'erano dati,
    # usiamo ffill() (forward fill) per copiare il valore dell'ultimo slot valido noto.
    df_resampled['differenza'] = df_resampled['differenza'].ffill()

    # 4. Feature Engineering: Creazione delle variabili indipendenti (X) per i modelli
    # Estraiamo l'ora del giorno (0-23)
    df_resampled['hour'] = df_resampled.index.hour
    # Estraiamo il giorno della settimana (0=Lunedì, 6=Domenica)
    df_resampled['dayofweek'] = df_resampled.index.dayofweek
    # Creiamo una variabile binaria (1/0) per indicare se è fine settimana (Sabato o Domenica)
    df_resampled['is_weekend'] = (df_resampled.index.dayofweek >= 5).astype(int)

    # Eliminiamo le primissime righe se contengono valori nulli impossibili da riempire
    df_resampled.dropna(inplace=True)

    return df_resampled

# Avviamo il caricamento
dataset_raw = carica_dataset_completo(FILE_NAME)

Mounted at /content/drive
 Cartella risultati creata: /content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/Risultati_Esperimenti/
Lettura del dataset completo in corso...


## 2: Sliding Window Dinamica
In questo blocco viene abbandonato il partizionamento statico (80/20). La funzione genera dinamicamente finestre di addestramento e test.
Utilizzando la modalità `test_on_end=True` e `train_days='auto'`, il sistema si ancora matematicamente all'ultimo giorno del dataset, calcolando tutto lo storico passato disponibile e generando iterazioni (folds) per testare rigorosamente le misurazioni più recenti.

In [ ]:
from datetime import timedelta

def sliding_window_generator(df, train_days='auto', test_days=1, step_days=1, max_folds=14, test_on_end=True):
    """
    Funzione di sliding window.
    Usa 'yield' per rilasciare un pezzo alla volta ottimizzando la memoria RAM.
    """
    # Controllo di sicurezza sull'indice
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("L'indice del dataset deve essere in formato Datetime!")

    start_date = df.index.min() # Data più vecchia
    end_date = df.index.max()   # Data più recente
    fold_counter = 1

    if test_on_end:
        # Vogliamo che l'ultimo giorno di test coincida con la fine del dataset
        # Calcoliamo quanti giorni totali durerà il periodo di test (es. 14 iterazioni * 1 giorno = 14 giorni)
        giorni_totali_test = max_folds * step_days

        # Calcoliamo la data in cui deve partire il PRIMO test
        inizio_primo_test = end_date - timedelta(days=giorni_totali_test)

        # Se richiesto, calcoliamo in automatico quanti giorni storici ci sono prima di quel test
        if train_days == 'auto':
            train_days = (inizio_primo_test - start_date).days

        # Impostiamo la data di partenza del primissimo blocco di addestramento
        current_start = inizio_primo_test - timedelta(days=train_days)

        # Controllo se il dataset è troppo corto per questa configurazione
        if current_start < start_date:
            raise ValueError(f"Dataset troppo corto per {max_folds} test e {train_days} storico!")
    else:
        # Modalità classica: parte dall'inizio del file
        current_start = start_date

    # Ciclo infinito che verrà interrotto dai 'break'
    while True:
        # CONDIZIONE DI STOP 1: Abbiamo raggiunto il numero massimo di finestre richieste (es. 14)
        if max_folds is not None and fold_counter > max_folds:
            break

        # Calcolo dei margini temporali del fold attuale
        train_end = current_start + timedelta(days=train_days)
        test_end = train_end + timedelta(days=test_days)

        # CONDIZIONE DI STOP 2: Il test supererebbe l'ultima riga del dataset
        if test_end > end_date:
            break

        # SLICING: Tagliamo fisicamente i dati.
        # Usiamo '<' per l'estremo destro per evitare sovrapposizioni tra train e test (Data Leakage)
        train_data = df[(df.index >= current_start) & (df.index < train_end)]
        test_data = df[(df.index >= train_end) & (df.index < test_end)]

        # Restituiamo i dati al blocco chiamante e mettiamo in pausa la funzione
        yield fold_counter, train_data, test_data

        # SCORRIMENTO: Spostiamo l'inizio in avanti di 'step_days' (es. 1 giorno) e ripartiamo
        current_start += timedelta(days=step_days)
        fold_counter += 1

##3: Evaluation Loop Modulare & Wrappers
Questo blocco definisce il motore di valutazione che calcola tempi e metriche offline/online, registrando l'errore commesso in ciascun singolo slot. Contiene inoltre i Wrappers degli algoritmi: Random Forest, Neural Prophet, Stacked LSTM e SARIMA.

In [ ]:
import time
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import statsmodels.api as sm
from neuralprophet import NeuralProphet
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
import logging

# Soppressione dei log di PyTorch e TensorFlow
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
tf.get_logger().setLevel('ERROR')

# =====================================================================
# VALUTAZIONE
# =====================================================================
def run_evaluation_loop_modulare(generatore, model_wrapper, path_salvataggio):
    """
    Riceve il i dati e il modello, li fa addestrare,
    calcola i tempi operativi e salva metriche e decadimento errori.
    """
    print(f" -> Loop: {model_wrapper.model_name}", end=" | ")
    tutti_i_risultati = []

    # Iteriamo su tutte le finestre prodotte dal generatore (Blocco 2)
    for fold, train_df, test_df in generatore:
        if train_df.empty or test_df.empty: continue
        y_test = test_df['differenza'].values

        # --- 1. FASE OFFLINE: ADDESTRAMENTO ---
        # Avviamo il cronometro
        start_train = time.perf_counter()
        # Il modello impara dal passato
        model_wrapper.train(train_df)
        # Stoppiamo il cronometro
        train_time = time.perf_counter() - start_train

        # --- 2. FASE ONLINE: INFERENZA (PREDIZIONE) ---
        start_pred = time.perf_counter()
        # Il modello cerca di indovinare il futuro
        y_pred = model_wrapper.predict(train_df, test_df)
        pred_time = time.perf_counter() - start_pred

        # --- 3. COMPILAZIONE METRICHE DI SISTEMA ---
        fold_result = {
            'modello': model_wrapper.model_name,
            'id_finestra': fold,
            'data_test': test_df.index.min().date(),
            'tempo_training_sec': round(train_time, 4),
            # Calcoliamo la latenza per la singola previsione (fondamentale per sistemi real-time)
            'latenza_ms_per_slot': round((pred_time / len(y_test)) * 1000, 4),
            # RMSE Globale della finestra (radice dell'errore quadratico medio)
            'rmse_globale': round(np.sqrt(mean_squared_error(y_test, y_pred)), 3),
            # MAE Globale (errore medio assoluto: sbaglia in media di X)
            'mae_globale': round(mean_absolute_error(y_test, y_pred), 3)
        }

        # --- 4. ANALISI DEL DECADIMENTO (Errore per singolo slot) ---
        # Invece di una media, salviamo quanto ha sbagliato nella prima mezz'ora, nella seconda, ecc.
        for i in range(len(y_test)):
            fold_result[f'errore_assoluto_slot_{i+1}'] = round(abs(y_test[i] - y_pred[i]), 3)

        tutti_i_risultati.append(fold_result)
        df_temp = pd.DataFrame([fold_result])

        # Salvataggio progressivo su file CSV.
        # mode='a' significa 'append': aggiunge riga per riga senza cancellare i dati precedenti.
        if not os.path.exists(path_salvataggio):
            df_temp.to_csv(path_salvataggio, index=False)
        else:
            df_temp.to_csv(path_salvataggio, mode='a', header=False, index=False)

    print("Completato.")
    return pd.DataFrame(tutti_i_risultati)

# =====================================================================
# WRAPPERS DEI MODELLI (Classi per standardizzare le chiamate train/predict)
# =====================================================================

class RandomForestWrapper:
    """Modello di Machine Learning Classico basato su alberi decisionali multipli"""
    def __init__(self):
        self.model_name = "RandomForest"
        # n_jobs=-1 usa tutti i processori disponibili per velocizzare
        self.model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
        self.features = ['hour', 'dayofweek', 'is_weekend']

    def train(self, train_df):
        self.model.fit(train_df[self.features].values, train_df['differenza'].values)

    def predict(self, train_df, test_df):
        # np.maximum(0, ...) forza le previsioni negative a zero (non esiste traffico negativo)
        return np.maximum(0, self.model.predict(test_df[self.features].values))


class NeuralProphetWrapper:
    """Modello avanzato di Facebook/PyTorch specializzato in serie storiche con stagionalità"""
    def __init__(self, epochs=50):
        self.model_name = "NeuralProphet"
        self.epochs = epochs

    def train(self, train_df):
        # NeuralProphet esige che le colonne si chiamino 'ds' (datetime) e 'y' (target)
        df_np = pd.DataFrame({'ds': train_df.index, 'y': train_df['differenza'].values})
        self.model = NeuralProphet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True, epochs=self.epochs)
        self.model.fit(df_np, freq="30min", progress=None)

    def predict(self, train_df, test_df):
        df_np = pd.DataFrame({'ds': train_df.index, 'y': train_df['differenza'].values})
        future = self.model.make_future_dataframe(df_np, periods=len(test_df))
        return np.maximum(0, self.model.predict(future)['yhat1'].values)


class StackedLSTMWrapper:
    """Modello Deep Learning (Rete Neurale Ricorrente) per imparare pattern temporali complessi"""
    def __init__(self, look_back=48, epochs=20):
        self.model_name = "Stacked_LSTM"
        self.look_back = look_back # Guarda alle ultime 48 mezz'ore (24 ore) per prevedere la successiva
        self.epochs = epochs
        # Le reti neurali lavorano bene solo se i numeri sono scalati tra 0 e 1
        self.scaler = MinMaxScaler(feature_range=(0, 1))

    def train(self, train_df):
        scaled = self.scaler.fit_transform(train_df[['differenza']].values)
        X, Y = [], []
        # Creiamo le sequenze: input (X) di 48 valori, target (Y) è il 49esimo valore
        for i in range(len(scaled) - self.look_back):
            X.append(scaled[i:(i + self.look_back), 0])
            Y.append(scaled[i + self.look_back, 0])

        # Reshape per Keras: [samples, time steps, features]
        X, Y = np.array(X), np.array(Y)
        X = np.reshape(X, (X.shape[0], X.shape[1], 1))

        # Architettura 'Stacked': due layer LSTM sovrapposti
        self.model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(self.look_back, 1)),
            Dropout(0.2), # Spegne a caso il 20% dei neuroni per evitare che impari a memoria (overfitting)
            LSTM(32, return_sequences=False),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1) # Output singolo: la previsione
        ])
        self.model.compile(loss='mse', optimizer='adam')
        self.model.fit(X, Y, epochs=self.epochs, batch_size=64, verbose=0)

    def predict(self, train_df, test_df):
        preds = []
        # Innesco: prendiamo gli ultimi 48 valori dell'addestramento
        batch = train_df[['differenza']].values[-self.look_back:]
        batch_scaled = self.scaler.transform(batch).reshape(1, self.look_back, 1)

        # Previsione Autoregressiva (prediciamo uno slot, lo accodiamo all'input, e prediciamo il successivo)
        for _ in range(len(test_df)):
            p_scaled = self.model.predict(batch_scaled, verbose=0)[0]
            # Riportiamo il valore scalato (es. 0.5) al numero reale di auto
            preds.append(max(0, self.scaler.inverse_transform([p_scaled])[0, 0]))
            # Aggiorniamo la finestra scartando il valore più vecchio e inserendo la nuova previsione
            batch_scaled = np.append(batch_scaled[:, 1:, :], [[[p_scaled[0]]]], axis=1)
        return np.array(preds)


class SARIMAXWrapper:
    """Modello puramente Statistico (AutoRegressive Integrated Moving Average)"""
    def __init__(self, order, seasonal_order):
        self.model_name = "SARIMA"
        # I parametri p,d,q e P,D,Q,s vengono iniettati dall'esterno (calcolati prima)
        self.order = order
        self.seasonal_order = seasonal_order

    def train(self, train_df):
        self.model = sm.tsa.SARIMAX(train_df['differenza'].values,
                                    order=self.order,
                                    seasonal_order=self.seasonal_order,
                                    enforce_stationarity=False,
                                    enforce_invertibility=False)
        self.fitted = self.model.fit(disp=False)

    def predict(self, train_df, test_df):
        return np.maximum(0, self.fitted.get_forecast(steps=len(test_df)).predicted_mean)

ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.
ERROR:NP.plotly:Importing plotly failed. Interactive plots will not work.


## 4: Esecuzione Batch Multi-Stazione
 Lo script esegue un loop su ogni stazione disponibile nel dataset. Per ogni stazione, esegue l'addestramento e il test sui 4 modelli, spingendo la previsione a **2 giorni futuri (48h)** su un set di test di **14 giorni (finestre)**.
*Nota: i parametri SARIMA sono pre-calcolati offline tramite script "una tantum" per prevenire colli di bottiglia.*

In [ ]:
if __name__ == "__main__":

    # =========================================================================
    # PARAMETRI DEL PROTOCOLLO SPERIMENTALE
    # =========================================================================
    TEST_GIORNI_FUTURI = 2 # Il modello prova a prevedere fino a 48 ore nel futuro
    MAX_ITERAZIONI = 14    # Isoliamo un periodo di test di 14 finestre (due settimane)

    # Estrae la lista unica di tutte le stazioni (1, 2, 3, 4, 5, 6, 7, 8, 9)
    stazioni_disponibili = dataset_raw['id_stazione'].unique()

    # =========================================================================
    # PARAMETRI SARIMA (Pre-calcolati con l'altro script per evitare blocchi RAM)
    # =========================================================================
   PARAMETRI_SARIMA_STAZIONI = {
        1: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        2: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        3: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        4: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        5: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        6: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        7: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        8: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
        9: {'order': (3, 0, 1), 'seasonal_order': (0, 1, 1, 48)},
    }

    # Stampa per tracciare cosa sta succedendo a schermo
    print("\n" + "#"*70)
    print(" INIZIO ELABORAZIONE BATCH MULTI-STAZIONE")
    print(f" Orizzonte di predizione : {TEST_GIORNI_FUTURI} giorni")
    print(f" Iterazioni di test      : {MAX_ITERAZIONI} finestre")
    print(f" Stazioni in analisi     : {len(stazioni_disponibili)}")
    print("#"*70 + "\n")

    # Ciclo Principale: iteriamo stazione per stazione
    for st_id in stazioni_disponibili:
        print(f"\n--- AVVIO PIPELINE: STAZIONE {st_id} ---")

        # Pulizia dati specifica per QUESTA stazione
        dataset_stazione = pre_processing_stazione(dataset_raw, st_id)

        # Controllo Dati: Se la stazione è guasta o ha meno di 30 giorni di storico, la saltiamo
        if len(dataset_stazione) < 48 * 30:
            print(f" -> Stazione {st_id} saltata (Dati insufficienti, meno di 30 giorni)")
            continue

        # =====================================================================
        # LANCIO DEI MODELLI IN CASCATA
        # =====================================================================

        # 1. Random Forest (Veloce e robusta)
        gen_rf = sliding_window_generator(dataset_stazione, train_days='auto', test_days=TEST_GIORNI_FUTURI, step_days=1, max_folds=MAX_ITERAZIONI, test_on_end=True)
        # Costruiamo il nome del file dinamicamente (es. risultati_RF_Staz_3.csv)
        path_rf = os.path.join(RESULTS_DIR, f"risultati_RF_Staz_{st_id}.csv")
        run_evaluation_loop_modulare(gen_rf, RandomForestWrapper(), path_rf)

        # 2. SARIMA (Rigore Statistico)
        # Verifichiamo se abbiamo i parametri per questa stazione nel dizionario
        if st_id in PARAMETRI_SARIMA_STAZIONI:
            p_order = PARAMETRI_SARIMA_STAZIONI[st_id]['order']
            p_seasonal = PARAMETRI_SARIMA_STAZIONI[st_id]['seasonal_order']

            gen_sarima = sliding_window_generator(dataset_stazione, train_days='auto', test_days=TEST_GIORNI_FUTURI, step_days=1, max_folds=MAX_ITERAZIONI, test_on_end=True)
            path_sarima = os.path.join(RESULTS_DIR, f"risultati_SARIMA_Staz_{st_id}.csv")
            run_evaluation_loop_modulare(gen_sarima, SARIMAXWrapper(order=p_order, seasonal_order=p_seasonal), path_sarima)
        else:
            print(f" -> SARIMA saltato (Parametri non forniti nel dizionario per Staz {st_id})")

        # 3. Neural Prophet (Focus sulle stagionalità complesse)
        gen_np = sliding_window_generator(dataset_stazione, train_days='auto', test_days=TEST_GIORNI_FUTURI, step_days=1, max_folds=MAX_ITERAZIONI, test_on_end=True)
        path_np = os.path.join(RESULTS_DIR, f"risultati_NP_Staz_{st_id}.csv")
        run_evaluation_loop_modulare(gen_np, NeuralProphetWrapper(epochs=50), path_np)

        # 4. Stacked LSTM (Deep Learning, capace di trovare logiche non lineari)
        gen_lstm = sliding_window_generator(dataset_stazione, train_days='auto', test_days=TEST_GIORNI_FUTURI, step_days=1, max_folds=MAX_ITERAZIONI, test_on_end=True)
        path_lstm = os.path.join(RESULTS_DIR, f"risultati_LSTM_Staz_{st_id}.csv")
        run_evaluation_loop_modulare(gen_lstm, StackedLSTMWrapper(epochs=20), path_lstm)

    print("\n!!! ELABORAZIONE COMPLETA CONCLUSA !!!")


######################################################################
 INIZIO ELABORAZIONE BATCH MULTI-STAZIONE
 Salvataggio file in     : /content/drive/MyDrive/Colab Notebooks/TIROCINIO/dataset_ricostruito/Risultati_Esperimenti/
 Orizzonte di predizione : 2 giorni
 Iterazioni di test      : 14 finestre
 Stazioni in analisi     : 9
######################################################################


--- AVVIO PIPELINE: STAZIONE 1 ---
 -> Loop: RandomForest | Completato.
 -> Loop: SARIMA | 

## 5: Analisi Globale del Decadimento dell'Errore
Da eseguire alla fine di tutte le elaborazioni. Lo script aggrega matematicamente i CSV generati su tutte le stazioni (Media Globale) per mappare come si deteriora l'affidabilità predittiva da 0 a 48 ore. La visualizzazione aiuta a decretare empiricamente ogni quanto il sistema reale necessita di una fase Offline di riaddestramento.

Legge tutti i CSV generati, media gli errori e stampa il grafico del ciclo di vita dei modelli.

In [ ]:
import matplotlib.pyplot as plt

def analizza_decadimento_globale(prefissi_modelli, labels_modelli):
    """
    Trova i CSV di ogni modello, aggrega le colonne degli errori per singolo slot e
    stampa un grafico temporale.
    """
    plt.figure(figsize=(14, 7))
    # Palette di colori leggibile e professionale per i 4 modelli
    colori = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

    dati_trovati = False

    # Per ogni modello (es. "risultati_RF_Staz_")...
    for idx, prefisso in enumerate(prefissi_modelli):
        # Usiamo glob per trovare TUTTI i file che matchano col prefisso (tutte le 9 stazioni)
        pattern = os.path.join(RESULTS_DIR, f"{prefisso}*.csv")
        file_modello = glob.glob(pattern)

        if len(file_modello) > 0:
            dati_trovati = True
            errori_tutte_stazioni = []

            # Leggiamo i file CSV di tutte le stazioni uno per uno
            for path_csv in file_modello:
                df_ris = pd.read_csv(path_csv)

                # Isoliamo solo le 96 colonne che si chiamano 'errore_assoluto_slot_X'
                colonne_errore = [col for col in df_ris.columns if col.startswith('errore_assoluto_slot_')]

                # Facciamo la media in VERTICALE: per ogni slot, qual è stato l'errore medio sulle 14 finestre di test?
                errore_stazione_per_slot = df_ris[colonne_errore].mean(axis=0).values
                errori_tutte_stazioni.append(errore_stazione_per_slot)

            # Ora facciamo la media GLOBALE su tutte le stazioni (es. l'errore medio della RF allo slot 1 su tutte le 9 stazioni)
            errore_medio_globale = np.mean(errori_tutte_stazioni, axis=0)

            # Creiamo l'asse X convertendo gli slot in Ore (es. slot 1 -> 0.5 ore, slot 2 -> 1.0 ore)
            asse_x_ore = np.arange(1, len(errore_medio_globale) + 1) * 0.5

            # Disegniamo la linea del modello sul grafico
            plt.plot(asse_x_ore, errore_medio_globale, label=f"{labels_modelli[idx]} (Media su {len(file_modello)} staz.)",
                     color=colori[idx % len(colori)], linewidth=2, marker='o', markersize=3)
        else:
            print(f"[!] Nessun file trovato per {labels_modelli[idx]}")

    # Formattazione grafica (Titoli, griglie, assi)
    if dati_trovati:
        plt.title('Decadimento Globale Affidabilità Predittiva nel Tempo (Orizzonte 48h)', fontsize=15, fontweight='bold')
        plt.xlabel('Orizzonte di Previsione (Ore future dal momento dell\'inferenza)', fontsize=12)
        plt.ylabel('Errore Medio Assoluto (MAE per slot) - Media Multi-Stazione', fontsize=12)

        # Disegniamo due linee verticali tratteggiate per separare visivamente i giorni
        plt.axvline(x=24, color='black', linestyle='--', alpha=0.6, label='Traguardo 24h')
        plt.axvline(x=48, color='black', linestyle='--', alpha=0.6, label='Traguardo 48h')

        plt.legend(fontsize=11)
        plt.grid(True, linestyle=':', alpha=0.7)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Impossibile generare il grafico: nessun CSV trovato in {RESULTS_DIR}")

# =====================================================================
# CHIAMATA DELLA FUNZIONE GRAFICA
# =====================================================================
# Le etichette dei file che vogliamo cercare e plottare
prefissi = [
    "risultati_RF_Staz_",
    "risultati_SARIMA_Staz_",
    "risultati_NP_Staz_",
    "risultati_LSTM_Staz_"
]
nomi = ["Random Forest", "SARIMA", "Neural Prophet", "Stacked LSTM"]

# Genera l'output a schermo
analizza_decadimento_globale(prefissi, nomi)